# Generic Chains Overview

## Simple Chain

<font color='green'>
The most elementary type of chain is known as a basic chain, which represents the simplest form of crafting a chain. <br>In this setup, there is only one Language Model (LLM) responsible for receiving an input prompt and using it for generating text.
<font>

In [1]:
#Please install the package as per your requirement :)
#!pip install openai==1.14.2
#!pip install langchain==0.1.13
#!pip install huggingface-hub==0.21.4
#!pip install langchain-openai==0.1.0

In [2]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

In [3]:
#The below import has been replaced by the later one
#from langchain.llms import OpenAI
from langchain_openai import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

In [4]:
llm = OpenAI()

In [5]:
prompt = PromptTemplate(
    input_variables=["place"],
    template="Best places to visit in {place}?",
)

In [6]:
chain = LLMChain(llm=llm, prompt=prompt)

# Run the chain only specifying the input variable.
# Recently langchain has replaced 'run' function with 'invoke'
print(chain.invoke("India"))

{'place': 'India', 'text': '\n\n1. Taj Mahal, Agra\n2. Varanasi, Uttar Pradesh\n3. Jaipur, Rajasthan\n4. Goa\n5. Kerala backwaters\n6. Ladakh, Jammu and Kashmir\n7. Hampi, Karnataka\n8. Ranthambore National Park, Rajasthan\n9. Darjeeling, West Bengal\n10. Jaisalmer, Rajasthan\n11. Amritsar, Punjab\n12. Munnar, Kerala\n13. Pushkar, Rajasthan\n14. Delhi\n15. Udaipur, Rajasthan\n16. Khajuraho, Madhya Pradesh\n17. Jodhpur, Rajasthan\n18. Dharamshala, Himachal Pradesh\n19. Srinagar, Jammu and Kashmir\n20. Hampi, Karnataka.'}


## Simple Sequential Chains

<font color='green'>
Sequential Chains involves making a series of consecutive calls to the language model.<br> This approach proves especially valuable when there is a need to utilize the output generated from one call as the input for another call.
<font>

In [7]:
from langchain.chains import SimpleSequentialChain

#from langchain.llms import HuggingFaceHub
#The above have been updated recently, so going forward we have to use the below :)

from langchain.llms import HuggingFaceEndpoint

In [8]:
template = """You have to suggest 5 best places to visit in {place}?

YOUR RESPONSE:
"""
prompt_template = PromptTemplate(
    input_variables=["place"], 
    template=template)

In [17]:
#HF_llm= HuggingFaceHub(repo_id = "google/flan-t5-large")
#The above 'HuggingFaceHub' class has been depreciated, so please use the below class'HuggingFaceEndpoint' 
#and the below mentioned model outperforms most of the available open source LLMs

HF_llm = HuggingFaceEndpoint(repo_id="mistralai/Mistral-7B-Instruct-v0.2") # Model link : https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2


In [18]:
llm = OpenAI()

In [19]:
place_chain = LLMChain(llm=llm, prompt=prompt_template)

In [20]:
template = """Given a list a places, please estimate the expenses to visit all of them in local currency and also the days needed
{expenses}

YOUR RESPONSE:
"""
prompt_template = PromptTemplate(
    input_variables=["expenses"],
    template=template)

In [21]:
llm = OpenAI()

In [22]:
expenses_chain = LLMChain(llm=llm, prompt=prompt_template)

In [23]:
final_chain = SimpleSequentialChain(chains=[place_chain, expenses_chain], verbose=True)

In [24]:
review = final_chain.invoke("India")



> Entering new SimpleSequentialChain chain...

1. Taj Mahal - Rs. 1300 ($18.18) - 1 day
2. Golden Temple - Free - 1 day
3. Red Fort - Rs. 35 ($0.49) - 1 day
4. Goa - Rs. 1500 ($20.93) - 2 days
5. Kerala backwaters - Rs. 5000 ($69.78) - 2 days
6. Jaipur - Rs. 1000 ($13.96) - 2 days
7. Varanasi - Rs. 500 ($6.98) - 1 day
8. Mumbai - Rs. 2000 ($27.92) - 2 days
9. Agra Fort - Rs. 650 ($9.06) - 1 day
10. Ajanta and Ellora Caves - Rs. 550 ($7.67) - 1 day

Total estimated expenses: Rs. 12,535 ($174.85)

Total estimated days needed: 14 days 

As a language model AI, I cannot visit places and only provide information. The estimated expenses for visiting all the places mentioned in the list will be around Rs. 12,535 ($174.85) in local currency and it will take around 14 days to cover all the places. This estimate is based on the entrance fees and approximate accommodation and transportation costs. Actual expenses may vary depending on individual preferences and travel style.

> Finished chain.
